In [ ]:
"""
Script para extraer entidades biomédicas de artículos de PubMed
Usa BioBERT/SciBERT para NER en lugar de BioGPT (más apropiado para esta tarea)
"""

import pandas as pd
from Bio import Entrez
import time
from typing import List, Dict
import re

# Configurar email para Entrez (REQUERIDO por NCBI)
Entrez.email = "tu_email@ejemplo.com"  # CAMBIAR ESTO

def obtener_articulo_pubmed(pmid: str) -> Dict:
    """
    Obtiene el título y abstract de un artículo de PubMed
    """
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        
        article = records['PubmedArticle'][0]['MedlineCitation']['Article']
        
        titulo = article.get('ArticleTitle', '')
        abstract = article.get('Abstract', {})
        
        # Extraer texto del abstract
        abstract_text = ''
        if abstract:
            abstract_parts = abstract.get('AbstractText', [])
            if isinstance(abstract_parts, list):
                abstract_text = ' '.join([str(part) for part in abstract_parts])
            else:
                abstract_text = str(abstract_parts)
        
        return {
            'pmid': pmid,
            'titulo': titulo,
            'abstract': abstract_text,
            'texto_completo': f"{titulo} {abstract_text}"
        }
    except Exception as e:
        print(f"Error obteniendo PMID {pmid}: {e}")
        return {
            'pmid': pmid,
            'titulo': '',
            'abstract': '',
            'texto_completo': ''
        }

def extraer_entidades_simple(texto: str) -> Dict:
    """
    Extracción simple usando patrones y diccionarios básicos
    Para producción, usar modelos como BioBERT, SciBERT o PubMedBERT
    """
    
    # Patrones básicos (esto es simplificado, usar modelos NER en producción)
    entidades = {
        'SPECIES': [],
        'TAXON': [],
        'GENE': [],
        'PROTEIN': [],
        'CHEMICAL': [],
        'DRUG': [],
        'DISEASE': [],
        'CONDITION': []
    }
    
    # Ejemplos de patrones simples (MEJORAR CON MODELOS NER REALES)
    # Especies comunes
    especies_patron = r'\b(human|humans|mouse|mice|rat|rats|E\. coli|Escherichia coli|SARS-CoV-2|HIV)\b'
    entidades['SPECIES'] = list(set(re.findall(especies_patron, texto, re.IGNORECASE)))
    
    # Genes (patrones típicos: mayúsculas, números)
    gene_patron = r'\b([A-Z]{2,}[0-9]+|[A-Z][a-z]+[0-9]+)\b'
    posibles_genes = re.findall(gene_patron, texto)
    entidades['GENE'] = list(set([g for g in posibles_genes if len(g) <= 10]))[:10]  # Limitar
    
    # Enfermedades comunes
    enfermedades = r'\b(cancer|diabetes|COVID-19|hypertension|infection|disease|syndrome|disorder)\b'
    entidades['DISEASE'] = list(set(re.findall(enfermedades, texto, re.IGNORECASE)))
    
    # Químicos/Drogas comunes
    quimicos = r'\b(glucose|protein|insulin|aspirin|acetaminophen|antibiotic|chemotherapy)\b'
    entidades['CHEMICAL'] = list(set(re.findall(quimicos, texto, re.IGNORECASE)))
    
    return entidades

def procesar_csv_pubmed(archivo_entrada: str, columna_pmid: str = 'pmid') -> pd.DataFrame:
    """
    Procesa un CSV con PMIDs y extrae entidades biomédicas
    """
    # Leer CSV
    df = pd.read_csv(archivo_entrada)
    
    if columna_pmid not in df.columns:
        raise ValueError(f"Columna '{columna_pmid}' no encontrada en el CSV")
    
    resultados = []
    
    print(f"Procesando {len(df)} artículos de PubMed...")
    
    for idx, pmid in enumerate(df[columna_pmid], 1):
        print(f"Procesando {idx}/{len(df)}: PMID {pmid}")
        
        # Obtener artículo
        articulo = obtener_articulo_pubmed(str(pmid))
        
        # Extraer entidades
        entidades = extraer_entidades_simple(articulo['texto_completo'])
        
        # Crear registro
        resultado = {
            'pmid': pmid,
            'titulo': articulo['titulo'],
            'species': ', '.join(entidades['SPECIES']),
            'taxon': ', '.join(entidades['TAXON']),
            'genes': ', '.join(entidades['GENE']),
            'proteins': ', '.join(entidades['PROTEIN']),
            'chemicals': ', '.join(entidades['CHEMICAL']),
            'drugs': ', '.join(entidades['DRUG']),
            'diseases': ', '.join(entidades['DISEASE']),
            'conditions': ', '.join(entidades['CONDITION'])
        }
        
        resultados.append(resultado)
        
        # Respetar límites de NCBI (3 requests/segundo)
        time.sleep(0.4)
    
    # Crear DataFrame con resultados
    df_resultados = pd.DataFrame(resultados)
    
    return df_resultados

# Ejemplo de uso con BioBERT (RECOMENDADO para producción)
def extraer_entidades_biobert(texto: str):
    """
    Función para usar BioBERT - requiere instalación adicional
    pip install transformers torch
    """
    try:
        from transformers import pipeline
        
        # Cargar pipeline NER con modelo biomédico
        ner_pipeline = pipeline(
            "ner",
            model="dmis-lab/biobert-base-cased-v1.1",
            tokenizer="dmis-lab/biobert-base-cased-v1.1",
            aggregation_strategy="simple"
        )
        
        # Extraer entidades
        entidades = ner_pipeline(texto[:512])  # Limitar longitud
        
        return entidades
    except ImportError:
        print("BioBERT no disponible. Instalar con: pip install transformers torch")
        return []

if __name__ == "__main__":
    # CONFIGURACIÓN
    archivo_entrada = "articles_with_pmcid.csv"  # Tu archivo CSV con los PMIDs
    archivo_salida = "pubmed_entidades_extraidas.csv"
    columna_pmid = "PMC_ID"  # Nombre de la columna con los PMIDs
    
    # Cambiar el email antes de ejecutar
    Entrez.email = "atauchimamani@gmail.com"  # OBLIGATORIO
    
    # Procesar
    try:
        df_resultados = procesar_csv_pubmed(archivo_entrada, columna_pmid)
        
        # Guardar resultados
        df_resultados.to_csv(archivo_salida, index=False)
        print(f"\n✓ Resultados guardados en: {archivo_salida}")
        print(f"✓ Total de artículos procesados: {len(df_resultados)}")
        
    except Exception as e:
        print(f"Error: {e}")

Procesando 607 artículos de PubMed...
Procesando 1/607: PMID PMC4136787
Procesando 2/607: PMID PMC3630201
Procesando 3/607: PMID PMC11988870
Procesando 4/607: PMID PMC7998608
Procesando 5/607: PMID PMC5587110
Procesando 6/607: PMID PMC8396460
Procesando 7/607: PMID PMC5666799
Procesando 8/607: PMID PMC5460236
Procesando 9/607: PMID PMC6222041
Procesando 10/607: PMID PMC6813909
Procesando 11/607: PMID PMC4095884
Procesando 12/607: PMID PMC3040128
Procesando 13/607: PMID PMC3177255
Procesando 14/607: PMID PMC11500582
Procesando 15/607: PMID PMC5387210
Procesando 16/607: PMID PMC4642138
Procesando 17/607: PMID PMC5387210
Procesando 18/607: PMID PMC2915878
Procesando 19/607: PMID PMC3901686
Procesando 20/607: PMID PMC6985101
Procesando 21/607: PMID PMC6387434
Procesando 22/607: PMID PMC6371294
Procesando 23/607: PMID PMC7072278
Procesando 24/607: PMID PMC8441986
Procesando 25/607: PMID PMC9400218
Procesando 26/607: PMID PMC9267413
Procesando 27/607: PMID PMC9576569
Procesando 28/607: PMID 

In [ ]:
dfg = pd.read_csv("articles_data_updated.csv")
dfg.head(20)

,pmid,titulo,species,taxon,genes,proteins,chemicals,drugs,diseases,conditions
0,PMC4136787,Hybridization characteristics of enzymatically...,mouse,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PMC3630201,Disposition and metabolism of indeloxazine hyd...,rats,NaN,NaN,NaN,glucose,NaN,NaN,NaN
2,PMC11988870,Shogaols from Zingiber officinale protect IMR3...,human,NaN,"IMR32, EC50",NaN,NaN,NaN,NaN,NaN
3,PMC7998608,Basic science curriculum during residency: jus...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PMC5587110,[Some problems of cryoophthalmology].,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PMC8396460,State-dependent block underlies the tissue spe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PMC5666799,Cancer and asbestos.,NaN,NaN,NaN,NaN,NaN,NaN,Cancer,NaN
7,PMC5460236,Effect of dextrans on o-toluidine methods for ...,NaN,NaN,NaN,NaN,glucose,NaN,NaN,NaN
8,PMC6222041,recA protein promoted DNA strand exchange.,NaN,NaN,NaN,NaN,protein,NaN,NaN,NaN
9,PMC6813909,Clinical merit of simulation images generated ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
